# Monoidal Jantzen Filtrations

this collection of sage programs implements the theory introduced by Fujita and Hernandez in https://arxiv.org/abs/2402.13544

use monoidal_jantzen_filtration([string]) to get the q-t character of the tensor product of fundamental modules with evaluation parameters according to the string.

In [24]:
from collections import Counter 
import itertools
var('q')
var('u')
FQ.<q> = FunctionField(QQbar)
R.<u> = FunctionField(FQ)


In [25]:
# this is a class implementing a module over U_q(\hat{sl2}), with the possibility of taking direct sums and tensor products

class AffineUrep:
    def __init__(self,dimension,E0,F0,K0,E1,F1,K1):
        self.dimension = dimension
        self.space = R^dimension  # Vector space over complex numbers
        if E0.nrows() != self.dimension or E0.ncols() != self.dimension:
            raise ValueError(f"E0 must be {self.dimension}x{self.dimension}")
        if E1.nrows() != self.dimension or E1.ncols() != self.dimension:
            raise ValueError(f"E1 must be {self.dimension}x{self.dimension}")
        if F0.nrows() != self.dimension or F0.ncols() != self.dimension:
            raise ValueError(f"F0 must be {self.dimension}x{self.dimension}")
        if F1.nrows() != self.dimension or F1.ncols() != self.dimension:
            raise ValueError(f"F1 must be {self.dimension}x{self.dimension}")
        if K0.nrows() != self.dimension or K0.ncols() != self.dimension:
            raise ValueError(f"K0 must be {self.dimension}x{self.dimension}")
        if K1.nrows() != self.dimension or K1.ncols() != self.dimension:
            raise ValueError(f"E must be {self.dimension}x{self.dimension}")
        if not (K0.is_invertible()):
            raise ValueError("K0 is not invertible")
        if not (K1.is_invertible()):
            raise ValueError("K1 is not invertible")
        if not ((K0*K1-K1*K0).is_zero() and 
                (K1*E1*K1.inverse() - q^2*E1).is_zero() and 
                (K0*E0*K0.inverse() - q^2*E0).is_zero() and
                (K0*F0*K0.inverse() - q^(-2)*F0).is_zero() and 
                (K1*F1*K1.inverse() - q^(-2)*F1).is_zero() and 
                (K0*E1*K0.inverse() - q^(-2)*E1).is_zero() and 
                (K0*F1*K0.inverse() - q^(2)*F1).is_zero() and 
                (K1*E0*K1.inverse() - q^(-2)*E0).is_zero() and 
                (K1*F0*K1.inverse() - q^(2)*F0).is_zero() and 
                (E0*F0-F0*E0 - (K0-K0.inverse())/(q-q^(-1))).is_zero() and 
                (E1*F1-F1*E1 - (K1-K1.inverse())/(q-q^(-1))).is_zero() and 
                (E0*F1-F1*E0).is_zero() and 
                (F0*E1-E1*F0).is_zero() and 
                (E0^3*E1-(q^3-q^(-3))/(q-q^(-1))*E0^2*E1*E0 + (q^3-q^(-3))/(q-q^(-1))*E0*E1*E0^2 - E1*E0^3).is_zero() and
                (E1^3*E0-(q^3-q^(-3))/(q-q^(-1))*E1^2*E0*E1 + (q^3-q^(-3))/(q-q^(-1))*E1*E0*E1^2 - E0*E1^3).is_zero() and
                (F0^3*F1-(q^3-q^(-3))/(q-q^(-1))*F0^2*F1*F0 + (q^3-q^(-3))/(q-q^(-1))*F0*F1*F0^2 - F1*F0^3).is_zero() and
                (F1^3*F0-(q^3-q^(-3))/(q-q^(-1))*F1^2*F0*F1 + (q^3-q^(-3))/(q-q^(-1))*F1*F0*F1^2 - F0*F1^3).is_zero()
                ):
            print(K0*K1-K1*K0) 
            print(K0*F0*K0.inverse() - q^(-2)*F0)
            print(K1*E1*K1.inverse() - q^2*E1)
            print(K0*E0*K0.inverse() - q^2*E0)
            print(K1*F1*K1.inverse() - q^(-2)*F1)
            print(K0*E1*K0.inverse() - q^(-2)*E1)
            print(K0*F1*K0.inverse() - q^(2)*F1)
            print(K1*E0*K1.inverse() - q^(-2)*E0)
            print(K1*F0*K1.inverse() - q^(2)*F0)
            print(E0*F0-F0*E0 - (K0-K0.inverse())/(q-q^(-1)))
            print(E1*F1-F1*E1 - (K1-K1.inverse())/(q-q^(-1)))
            print("Commutators zero")
            print(E0*F1-F1*E0)
            print(F0*E1-E1*F0)
            print("Serre relations")
            print(E0^3*E1-(q^3-q^(-3))/(q-q^(-1))*E0^2*E1*E0 + (q^3-q^(-3))/(q-q^(-1))*E0*E1*E0^2 - E1*E0^3)
            print(E1^3*E0-(q^3-q^(-3))/(q-q^(-1))*E1^2*E0*E1 + (q^3-q^(-3))/(q-q^(-1))*E1*E0*E1^2 - E0*E1^3)
            print(F0^3*F1-(q^3-q^(-3))/(q-q^(-1))*F0^2*F1*F0 + (q^3-q^(-3))/(q-q^(-1))*F0*F1*F0^2 - F1*F0^3)
            print(F1^3*F0-(q^3-q^(-3))/(q-q^(-1))*F1^2*F0*F1 + (q^3-q^(-3))/(q-q^(-1))*F1*F0*F1^2 - F0*F1^3)
            raise ValueError("The relations do not hold!")
        self.E0 = E0
        self.F0 = F0
        self.K0 = K0
        self.E1 = E1
        self.F1 = F1
        self.K1 = K1
        
    def E0_mult(self, vector):
        return self.E0 * vector
    def F0_mult(self, vector):
        return self.F0 * vector
    def K0_mult(self, vector):
        return self.K0 * vector
    def K0inv_mult(self, vector):
        return (self.K0).inverse() * vector
    def E1_mult(self, vector):
        return self.E1 * vector
    def F1_mult(self, vector):
        return self.F1 * vector
    def K1_mult(self, vector):
        return self.K1 * vector
    def K1inv_mult(self, vector):
        return (self.K1).inverse() * vector
        
    def basis_element(self, i):
        """Get the i-th standard basis element"""
        v = vector(QQ, [0]*self.dimension)
        v[i] = 1
        return v
    def print_matrices(self):
        print("This is a U-rep of dimension " + str(self.dimension) + ". E0,F0,K0,E1,F1,K1 act by ")
        print("E0:")
        print(self.E0)
        print("F0:")
        print(self.F0)
        print("K0:")
        print(self.K0)
        print("E1:")
        print(self.E1)
        print("F1:")
        print(self.F1)
        print("K1:")
        print(self.K1)
                    
def tensor_maps_of_AffineUreps(space1, space2):
    
    n1, n2 = space1.dimension, space2.dimension
    tensor_dimension = n1 * n2
    
    # Create basis labels for tensor product: (i,j) means e_i ⊗ f_j
    tensor_basis_labels = [(i, j) for i in range(n1) for j in range(n2)]
    # print(tensor_basis_labels)
    # Initialize the matrices for E,F,K acting on the tensor product
    tensor_map_E0 = matrix(R, tensor_dimension, tensor_dimension)
    tensor_map_F0 = matrix(R, tensor_dimension, tensor_dimension)
    tensor_map_K0 = matrix(R, tensor_dimension, tensor_dimension)
    # F0or each basis element e_i ⊗ f_j of the tensor product
    for col_idx, (i, j) in enumerate(tensor_basis_labels):
        # precompute all necessary actions on the base spaces
        E0_ei = space1.E0_mult(space1.basis_element(i))
        E0_fj = space2.E0_mult(space2.basis_element(j))
        F0_ei = space1.F0_mult(space1.basis_element(i))
        F0_fj = space2.F0_mult(space2.basis_element(j))
        K0_ei = space1.K0_mult(space1.basis_element(i))
        K0_fj = space2.K0_mult(space2.basis_element(j))
        K0inv_ei = space1.K0inv_mult(space1.basis_element(i))
        K0inv_fj = space2.K0inv_mult(space2.basis_element(j))
        # Compute K0(e_i ⊗ f_j) = K0(e_i) ⊗ K0(f_j)
        for k in range(n1):
            for l in range(n2):
              #  if K0_ei[k] != 0 and K0_fj[l] != 0:
                    row_idx = tensor_basis_labels.index((k, l))
                    tensor_map_K0[row_idx, col_idx] += K0_ei[k] * K0_fj[l]
        # Compute F0(e_i ⊗ f_j) = F0(e_i) ⊗ K0inv(f_j) + e_i ⊗ F0(f_j)
        # first term
        for k in range(n1):
            for l in range(n2):
              #  if F0_ei[k] != 0 and K0inv_fj[l] != 0:
                    row_idx = tensor_basis_labels.index((k, l))
                    tensor_map_F0[row_idx, col_idx] += K0_ei[k] *F0_fj[l] #replace by K0inv
        # second term
        for k in range(n1):
           # if F0_fj[l] != 0:
                row_idx = tensor_basis_labels.index((k,j))
                # print(i,j,col_idx,k,l)
                tensor_map_F0[row_idx, col_idx] += F0_ei[k]
        # Compute E0(e_i ⊗ f_j) = K0(e_i) ⊗ E0(f_j) + E0(e_i) ⊗ 1(f_j)
        # first term
        for k in range(n1):
            for l in range(n2):
            #    if K0_ei[k] != 0 and E0_fj[l] != 0:
                    row_idx = tensor_basis_labels.index((k, l))
                    tensor_map_E0[row_idx, col_idx] += E0_ei[k] * K0inv_fj[l] #replace by K0
        # second term
        for l in range(n2):
           # if E0_ei[k] != 0:
                row_idx = tensor_basis_labels.index((i,l))
                tensor_map_E0[row_idx, col_idx] += E0_fj[l] 
                
    tensor_map_E1 = matrix(R, tensor_dimension, tensor_dimension)
    tensor_map_F1 = matrix(R, tensor_dimension, tensor_dimension)
    tensor_map_K1 = matrix(R, tensor_dimension, tensor_dimension)
    # F1or each basis element e_i ⊗ f_j of the tensor product
    for col_idx, (i, j) in enumerate(tensor_basis_labels):
        # precompute all necessary actions on the base spaces
        E1_ei = space1.E1_mult(space1.basis_element(i))
        E1_fj = space2.E1_mult(space2.basis_element(j))
        F1_ei = space1.F1_mult(space1.basis_element(i))
        F1_fj = space2.F1_mult(space2.basis_element(j))
        K1_ei = space1.K1_mult(space1.basis_element(i))
        K1_fj = space2.K1_mult(space2.basis_element(j))
        K1inv_ei = space1.K1inv_mult(space1.basis_element(i))
        K1inv_fj = space2.K1inv_mult(space2.basis_element(j))
        # Compute K1(e_i ⊗ f_j) = K1(e_i) ⊗ K1(f_j)
        for k in range(n1):
            for l in range(n2):
              #  if K1_ei[k] != 0 and K1_fj[l] != 0:
                    row_idx = tensor_basis_labels.index((k, l))
                    tensor_map_K1[row_idx, col_idx] += K1_ei[k] * K1_fj[l]
        # Compute F1(e_i ⊗ f_j) = F1(e_i) ⊗ K1inv(f_j) + e_i ⊗ F1(f_j)
        # first term
        for k in range(n1):
            for l in range(n2):
              #  if F1_ei[k] != 0 and K1inv_fj[l] != 0:
                    row_idx = tensor_basis_labels.index((k, l))
                    tensor_map_F1[row_idx, col_idx] += K1_ei[k] * F1_fj[l] #replace by K1_inv
        # second term
        for k in range(n1):
          #  if F1_fj[l] != 0:
                row_idx = tensor_basis_labels.index((k,j))
                # print(i,j,col_idx,k,l)
                tensor_map_F1[row_idx, col_idx] += F1_ei[k]
        # Compute E1(e_i ⊗ f_j) = K1(e_i) ⊗ E1(f_j) + E1(e_i) ⊗ 1(f_j)
        # first term
        for k in range(n1):
            for l in range(n2):
           #     if K1_ei[k] != 0 and E1_fj[l] != 0:
                    row_idx = tensor_basis_labels.index((k, l))
                    tensor_map_E1[row_idx, col_idx] += E1_ei[k] * K1inv_fj[l] #replace by K1
        # second term
        for l in range(n2):
           # if E1_ei[k] != 0:
                row_idx = tensor_basis_labels.index((i,l))
                tensor_map_E1[row_idx, col_idx] += E1_fj[l]
                
    return tensor_map_E0, tensor_map_F0, tensor_map_K0, tensor_map_E1, tensor_map_F1, tensor_map_K1

def tensor_product_of_AffineUreps(space1, space2):
    n1, n2 = space1.dimension, space2.dimension
    tensor_dimension = n1 * n2
    tensor_map_E0, tensor_map_F0, tensor_map_K0, tensor_map_E1, tensor_map_F1, tensor_map_K1 = tensor_maps_of_AffineUreps(space1,space2)
    result = AffineUrep(tensor_dimension, tensor_map_E0, tensor_map_F0, tensor_map_K0, tensor_map_E1, tensor_map_F1, tensor_map_K1)
    return result

def sum_maps_of_AffineUreps(space1, space2):

    # Initialize the matrices for E,F,K acting on the tensor product
    sum_map_E0 = (space1.E0).block_sum(space2.E0)
    sum_map_F0 = (space1.F0).block_sum(space2.F0)
    sum_map_K0 = (space1.K0).block_sum(space2.K0)
    sum_map_E1 = (space1.E1).block_sum(space2.E1)
    sum_map_F1 = (space1.F1).block_sum(space2.F1)
    sum_map_K1 = (space1.K1).block_sum(space2.K1)

    return sum_map_E0, sum_map_F0, sum_map_K0, sum_map_E1, sum_map_F1, sum_map_K1

def sum_of_AffineUreps(space1, space2):
    
    n1, n2 = space1.dimension, space2.dimension
    sum_dimension = n1 + n2

    sum_map_E0, sum_map_F0, sum_map_K0, sum_map_E1, sum_map_F1, sum_map_K1 = sum_maps_of_AffineUreps(space1,space2)
    result = AffineUrep(sum_dimension, sum_map_E0, sum_map_F0, sum_map_K0, sum_map_E1, sum_map_F1, sum_map_K1)
    return result

def dual_maps_of_AffineUrep(space):
    n = space.dimension
    dual_E0 = matrix(R, n,n)
    dual_F0 = matrix(R, n,n)
    dual_K0 = matrix(R, n,n)
    dual_E1 = matrix(R, n,n)
    dual_F1 = matrix(R, n,n)
    dual_K1 = matrix(R, n,n)
    #  Compute E(f)(v) = f(S(E)v) = -f(K^{-1}Ev)
    for row in range(0,n):
        for col in range(0,n):
            dual_E0[col,row] = -space.basis_element(row).dot_product(space.K0inv_mult(space.E0_mult(space.basis_element(col))))
            dual_F0[col,row] = -space.basis_element(row).dot_product(space.F0_mult(space.K0_mult(space.basis_element(col))))
            dual_K0[col, row] = space.basis_element(row).dot_product(space.K0inv_mult(space.basis_element(col)))
            dual_E1[col,row] = -space.basis_element(row).dot_product(space.K1inv_mult(space.E1_mult(space.basis_element(col))))
            dual_F1[col,row] = -space.basis_element(row).dot_product(space.F1_mult(space.K1_mult(space.basis_element(col))))
            dual_K1[col, row] = space.basis_element(row).dot_product(space.K1inv_mult(space.basis_element(col)))
    return dual_E0, dual_F0, dual_K0, dual_E1, dual_F1, dual_K1

def dual_AffineUrep(space):
    n = space.dimension
    dual_E0, dual_F0, dual_K0, dual_E1, dual_F1, dual_K1 = dual_maps_of_AffineUrep(space)
    return AffineUrep(n, dual_E0, dual_F0, dual_K0, dual_E1, dual_F1, dual_K1)

def matrix_E(n):
    E = matrix(R,n+1,n+1,0)  
    # Fill the superdiagonal with α values
    for r in range(1, n+1):
        alpha_r = (q^r - q^(-r))/(q - q^(-1))^2 * (q^(n-r+1) - q^(-n+r-1))
        E[r-1, r] = alpha_r    
    return E

def matrix_F(n):
    F = matrix(R,n+1,n+1,0)    
    # Fill the subdiagonal with 1's
    for i in range(1, n+1):
        F[i, i-1] = 1   
    return F

def matrix_K(n):
    K = matrix(R,n+1,n+1,0)
    # Fill the superdiagonal with powers of q
    for i in range(n+1):
        K[i, i] = q^(n - 2*i)
    return K

def sheared_Urep(space):
    n = space.dimension
    Z = zero_matrix(R,n)
    E1 = space.E1
    shear_F0 = block_matrix([[E1,-E1],[Z,E1]])
    shear_E1 = block_matrix([[E1,Z],[Z,E1]])
    F1 = space.F1
    shear_E0 = block_matrix([[F1,F1],[Z,F1]])
    shear_F1 = block_matrix([[F1,Z],[Z,F1]])
    K1 = space.K1
    K0 = space.K0
    shear_K0 = block_matrix([[K0,Z],[Z,K0]])
    shear_K1 = block_matrix([[K1,Z],[Z,K1]])
    return AffineUrep(2*n, shear_E0, shear_F0, shear_K0, shear_E1, shear_F1, shear_K1)

def eval_rep(n,a):
    if a == 0:
        raise ValueError("The evaluation rep needs a not equal to zero.")
    return AffineUrep(n+1,q^(-1)*a*matrix_F(n),q*a^(-1)*matrix_E(n),matrix_K(n).inverse(),matrix_E(n),matrix_F(n),matrix_K(n))

In [26]:

def R_matrix(a,b):
    # this is the R-matrix from V_1(a) \tensor V_1(b) to V_1(b) \tensor V_1(a)
    E = zero_matrix(R,4,4)  
    E[0,0] = 1
    E[3,3] = 1
    E[1,1] = u*(1-q^(-2))/(u-q^(b-a-2))
    E[2,1] = q^(-1)*(u-q^(b-a))/(u-q^(b-a-2))
    E[1,2] = q^(-1)*(u-q^(b-a))/(u-q^(b-a-2))
    E[2,2] = q^(b-a)*(1-q^(-2))/(u-q^(b-a-2))
    return E

def _bubble_sort_transpositions(a, descending=True):
    """
    Return the sequence of adjacent-swap positions (0-indexed: position i
    means indices i, i+1) that bubble-sorts `a` into descending
    (or ascending, if descending=False) order, along with the resulting
    sorted list. Positions are yielded together with the values being
    swapped at the time of the swap.
    """
    a = list(a)
    n = len(a)
    swaps = []  # list of (position, val_left, val_right) at time of swap
    changed = True
    while changed:
        changed = False
        for i in range(n - 1):
            cond = (a[i] < a[i + 1]) if descending else (a[i] > a[i + 1])
            if cond:
                swaps.append((i, a[i], a[i + 1]))
                a[i], a[i + 1] = a[i + 1], a[i]
                changed = True
    return swaps


def _embed_4x4(M, pos, n, base_ring):
    """
    Embed a 4x4 matrix M acting on tensor slots (pos, pos+1)
    into a 2^n x 2^n matrix via id (x) ... (x) M (x) ... (x) id.
    """
    mats = []
    i = 0
    while i < n:
        if i == pos:
            mats.append(M)
            i += 2
        else:
            mats.append(identity_matrix(base_ring, 2))
            i += 1
    result = mats[0]
    for m in mats[1:]:
        result = result.tensor_product(m)
    return result


def _R_matrix_product(a, descending, base_ring, n):
    """
    Build the R-matrix obtained by starting from the identity and,
    for each adjacent transposition used to bubble-sort `a` into
    descending (or ascending) order, left-multiplying by
        id (x) ... (x) R_matrix(a_{i-1}, a_i) (x) ... (x) id
    where a_{i-1}, a_i are the values at the swapped positions at the
    time of the swap.
    """
    swaps = _bubble_sort_transpositions(a, descending=descending)

    total = identity_matrix(base_ring, 2**n)
    for pos, val_left, val_right in swaps:
        if descending: #in this case the bubblesort algorithm gives the indices in the wrong order
            Rmat = R_matrix(val_right, val_left)
        else:
            Rmat = R_matrix(val_left, val_right)
        R_embed = _embed_4x4(Rmat, pos, n, base_ring)

        total = R_embed * total
    return total


def R_and_inverse_matrices(a, base_ring=None):
    """
    Given a string/list a = [a_1, ..., a_n]:

    - R_dec: built by bubble-sorting `a` into descending order;
      each swap of positions (i-1, i) contributes a left-multiplication by
          id (x) ... (x) R_matrix(a_{i-1}, a_i) (x) ... (x) id.

    - R_asc_inv: the inverse of the matrix built the same way, but
      bubble-sorting the *original* `a` into ascending order
      instead.
    """
    a = list(a)
    n = len(a)

    if base_ring is None:
        if n >= 2:
            base_ring = R_matrix(a[0], a[1]).base_ring()
        else:
            base_ring = QQ

    R_dec = _R_matrix_product(a, descending=True, base_ring=base_ring, n=n)
    R_asc = _R_matrix_product(a, descending=False, base_ring=base_ring, n=n)
    R_asc_inv = R_asc.inverse()

    return R_dec, R_asc_inv



In [29]:
from sage.all import infinity

def valuation_at_u1(f, u):

    if f == 0:
        return infinity
    return f.valuation(u-1)

def local_smith_form_at_u1(A, u):
    """
    R: invertible N x N matrix over F = Frac(QQbar[...,u,...]).

    Computes P, Q (invertible over the local ring O of functions regular
    at u=1) and a diagonal matrix D = P*R*Q, by pivoting at each step on
    the entry of globally minimal (u-1)-valuation in the active submatrix
    and clearing its row and column exactly (field division).

    Returns (vals, Q, P, D) where vals[j] = valuation_at_u1(D[j,j], u).
    """
    F = A.base_ring()
    N = A.nrows()
    assert A.ncols() == N, "R must be square"

    M = matrix(F, A)          # mutable working copy
    P = identity_matrix(F, N) # row-op accumulator
    Q = identity_matrix(F, N) # column-op accumulator

    vals = [None] * N
    active_rows = list(range(N))
    active_cols = list(range(N))

    while active_rows:
        # find globally minimal-valuation nonzero entry in active submatrix
        best = None
        for i in active_rows:
            for j in active_cols:
                v = M[i, j]
                if v != 0:
                    val = valuation_at_u1(v, u)
                    if best is None or val < best[0]:
                        best = (val, i, j)
        if best is None:
            raise ValueError(
                "Remaining submatrix is identically zero: R is not "
                "invertible over F, so a full valuation-graded basis "
                "of this type does not exist."
            )
        val, pi, pj = best
        pivot = M[pi, pj]

        # clear column pj (all other active rows) via row ops
        for i in active_rows:
            if i == pi:
                continue
            c = -M[i, pj] / pivot
            if c != 0:
                M.add_multiple_of_row(i, pi, c)
                P.add_multiple_of_row(i, pi, c)

        # clear row pi (all other active columns) via column ops
        for j in active_cols:
            if j == pj:
                continue
            c = -M[pi, j] / pivot
            if c != 0:
                M.add_multiple_of_column(j, pj, c)
                Q.add_multiple_of_column(j, pj, c)

        M.swap_columns(pi,pj)
        Q.swap_columns(pi,pj)
        vals[pi] = val
        active_rows.remove(pi)
        active_cols.remove(pi)

    #assert M == P*A*Q
    #assert M.is_diagonal()
    
    return vals, Q, P, M

def valuation_subspaces(A, u, descending, verify=True):
    """
    Given an invertible matrix R in variables q,u, return a
    dict {r: basis_matrix} where basis_matrix is an N x k_r matrix whose
    columns form a basis (entries regular at u=1) of the subspace V_r of
    the codomain all of whose entries have
    (u-1)-valuation >= r

    """
    vals, Q, P, D = local_smith_form_at_u1(A, u) # compute smith form

    #extract image basis
    if descending:
        M = (Q.subs(u=1)).transpose().inverse()
    else:
        M = (P.subs(u=1)).inverse()

    # create dictionary of rows of M
    N = A.nrows()
    groups = {}
    for j in range(N):
        groups.setdefault(vals[j], []).append(j)

    subspaces = {}
    prev_matrix = matrix(FQ,N,0,[])
    for r, cols in sorted(groups.items()):
        B = matrix(FQ, N, len(cols),
                    lambda i, k: M[i, cols[k]])
        subspaces[r] = prev_matrix.augment(B)
        prev_matrix = subspaces[r]

    return subspaces
    
def intersection_filtration(D1, D2, base_ring=None):
    """
    Given two dictionaries D1, D2 with integer keys and matrix values
    (columns of each matrix span a subspace of a common ambient space
    of dimension N), construct the descending filtration

        V_r = sum_{(k1,k2): -k1-k2 >= r}  ( colspan(D1[k1]) ∩ colspan(D2[k2]) )

    Returns a dict {r: basis_matrix}, keyed by every "breakpoint" value
    r = -k1-k2 achieved by some pair (k1,k2) in D1 x D2, sorted in
    descending order of r. basis_matrix is an N x dim(V_r) matrix whose
    columns form a basis of V_r.

    (V_r for r strictly between two consecutive breakpoints equals the
    subspace at the nearest larger breakpoint, since the filtration is
    constant there; only breakpoints are returned.)
    """
    if not D1 or not D2:
        raise ValueError("D1 and D2 must both be nonempty dictionaries.")

    sample = next(iter(D1.values()))
    N = sample.nrows()
    if base_ring is None:
        base_ring = sample.base_ring()

    ambient = VectorSpace(base_ring, N)

    # column spans of each piece, computed once
    span1 = {k: ambient.subspace((M.columns())) for k, M in D1.items()}
    span2 = {k: ambient.subspace((M.columns())) for k, M in D2.items()}

    # all pairwise intersections, tagged by their score s = -k1-k2
    pair_spaces = {}  # score -> list of subspaces (one per pair with that score)
    for k1, S1 in span1.items():
        for k2, S2 in span2.items():
            s = -k1 - k2
            inter = S1.intersection(S2)
            pair_spaces.setdefault(s, []).append(inter)

    # breakpoints sorted descending
    scores_desc = sorted(pair_spaces.keys(), reverse=True)

    filtration = {}
    running = ambient.subspace([])  # zero subspace
    for r in scores_desc:
        for sp in pair_spaces[r]:
            running = running + sp
        # running now equals sum over all pairs with score >= r
        basis_cols = running.basis()
        dim = len(basis_cols)
        if dim == 0:
            B = matrix(base_ring, N, 0)
        else:
            B = matrix(base_ring, N, dim, lambda i, j: basis_cols[j][i])
        filtration[r] = B
    return filtration

def _extend_basis(sub_basis_vectors, V_full, ambient):
    """
    Given a list of vectors sub_basis_vectors forming a basis of a
    subspace of V_full, return a list of extra vectors from V_full's
    basis that extend sub_basis_vectors to a full basis of V_full.
    """
    F = ambient.base_ring()
    current = list(sub_basis_vectors)
    target_dim = V_full.dimension()
    extra = []
    if len(current) == target_dim:
        return extra
    for v in V_full.basis():
        test = current + [v]
        M = matrix(F, test)
        if M.rank() > len(current):
            current.append(v)
            extra.append(v)
        if len(current) == target_dim:
            break
    if len(current) != target_dim:
        print(sub_basis_vectors)
        raise RuntimeError("Failed to extend basis")
    return extra


def graded_action_matrices(filtration, matrices, base_ring=None):
    """
    filtration: dict {r: basis_matrix} as produced by intersection_filtration,
        i.e. a descending chain V_{r_1} ⊇ V_{r_2} ⊇ ... (r_1 > r_2 > ...).
    matrices: list [M_1, ..., M_n] of N x N matrices (N = ambient dimension),
        each assumed to preserve every V_r in the filtration (M_i(V_r) ⊆ V_r).

    For each step r in the filtration, let V_prev be the next subspace down
    in the chain (or the zero subspace, if r is the last/smallest step).
    Returns a dict {r: [M_1^(r), ..., M_n^(r)]} where M_i^(r) is the
    matrix of the induced action of M_i on the quotient V_r / V_prev,
    expressed in the basis given by a chosen set of coset representatives.
    """
    if not filtration:
        raise ValueError("filtration must be nonempty.")

    rs = sorted(filtration.keys(), reverse=True)
    sample = next(iter(filtration.values()))
    N = sample.nrows()
    if base_ring is None:
        base_ring = sample.base_ring()

    ambient = VectorSpace(base_ring, N)
    subspaces = {r: ambient.subspace(filtration[r].columns()) for r in rs}
    zero_space = ambient.subspace([])

    result = {}
    for idx, r in enumerate(rs):
        V_r = subspaces[r]
        V_prev = subspaces[rs[idx - 1]] if idx > 0 else zero_space
        sub_basis = list(V_prev.basis())
        extra = _extend_basis(sub_basis, V_r, ambient)
        full_basis = sub_basis + extra
        m = len(extra)
        k = len(sub_basis)

        if m == 0:
            # quotient is zero-dimensional
            result[r] = [matrix(base_ring, 0, 0) for _ in matrices]
            continue

        B = matrix(base_ring, N, len(full_basis),
                    lambda i, j: full_basis[j][i])

        induced_matrices = []
        for M in matrices:
            cols = []
            for f in extra:
                fv = vector(base_ring, f)
                w = M * fv
                try:
                    x = B.solve_right(w)
                except ValueError:
                    raise ValueError(
                        f"M={M} does not preserve V_r at r={r}: image of a "
                        "quotient basis vector does not lie in V_r."
                    )
                cols.append(x[k:])  # drop V_prev-part, keep quotient-part

            M_r = matrix(base_ring, m, m, lambda i, j: cols[j][i])
            induced_matrices.append(M_r)
        rep_r = AffineUrep(m,induced_matrices[0],induced_matrices[1],induced_matrices[2],induced_matrices[3],induced_matrices[4],induced_matrices[5]) 
        print("The following representation is the coefficient of t^"+(r).str())
        rep_r.print_matrices()
    return result

def monoidal_jantzen_filtration(a):
    V = eval_rep(1,q^(a[0]))
    for spec_par in range(1,len(a)):
        # want to compute filtration on V
        V = tensor_product_of_AffineUreps(V, eval_rep(1,q^(a[spec_par])))
    #V.print_matrices()
    matrices_V = [V.E0,V.F0,V.K0,V.E1,V.F1,V.K1]
    print("computed rep")
    R_dec, R_asc_inv = R_and_inverse_matrices(a)
    print("computed R matrices")

    val_decomp_dec = valuation_subspaces(R_dec,u,True,verify=False)
    print("computed descending decomposition")
    val_decomp_asc = valuation_subspaces(R_asc_inv,u,False, verify=False)
    print("computed ascending decomposition")
    F = intersection_filtration(val_decomp_dec, val_decomp_asc)


    print('this is the filtration')
    print(F)
    print('this is the character, by coefficients of t^r')
    graded_action_matrices(F, matrices_V, base_ring=None)

        

In [30]:
monoidal_jantzen_filtration([3,3,1])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{2: [ 0  0]
[ 1  0]
[-q  0]
[ 0  0]
[ 0  0]
[ 0  1]
[ 0 -q]
[ 0  0], 0: [1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1]}
this is the character, by coefficients of t^r
The following representation is the coefficient of t^2
This is a U-rep of dimension 2. E0,F0,K0,E1,F1,K1 act by 
E0:
[  0   0]
[q^2   0]
F0:
[    0 1/q^2]
[    0     0]
K0:
[1/q   0]
[  0   q]
E1:
[0 1]
[0 0]
F1:
[0 0]
[1 0]
K1:
[  q   0]
[  0 1/q]
The following representation is the coefficient of t^0
This is a U-rep of dimension 6. E0,F0,K0,E1,F1,K1 act by 
E0:
[      0       0       0       0       0       0]
[q^2 + 1       0       0       0       0       0]
[      0       q       0       0       0       0]
[    q^4       0       0       0       0       0]
[      0     q^2       0 q^2 + 1       0       0

In [31]:
monoidal_jantzen_filtration([3,1,3])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{1: [], 0: [1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1], -1: [1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1]}
this is the character, by coefficients of t^r
The following representation is the coefficient of t^0
This is a U-rep of dimension 8. E0,F0,K0,E1,F1,K1 act by 
E0:
[  0   0   0   0   0   0   0   0]
[q^2   0   0   0   0   0   0   0]
[  q   0   0   0   0   0   0   0]
[  0 1/q q^2   0   0   0   0   0]
[q^4   0   0   0   0   0   0   0]
[  0 q^2   0   0 q^2   0   0   0]
[  0   0 q^2   0   q   0   0   0]
[  0   0   0   1   0 1/q q^2   0]
F0:
[    0 1/q^4   1/q     0 1/q^2     0     0     0]
[    0     0     0   1/q     0 1/q^2     0     0]
[    0     0     0 1/q^2     0   

In [8]:
monoidal_jantzen_filtration([1,3,3])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{0: [  1   0   0   0   0   0]
[  0   1   0   0   0   0]
[  0   0   1   0   0   0]
[  0   0   0   1   0   0]
[  0   0 1/q   0   0   0]
[  0   0   0 1/q   0   0]
[  0   0   0   0   1   0]
[  0   0   0   0   0   1], -2: [1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1]}
this is the character, by coefficients of t^r
The following representation is the coefficient of t^0
This is a U-rep of dimension 6. E0,F0,K0,E1,F1,K1 act by 
E0:
[            0             0             0             0             0             0]
[          q^2             0             0             0             0             0]
[          q^3             0             0             0             0             0]
[            0             q           q^2             0             0             0]
[       

In [11]:
monoidal_jantzen_filtration([5,3,1])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{1: [   0    0    0    0]
[   1    0    0    0]
[   0    1    0    0]
[   0    0    1    0]
[-q^2   -q    0    0]
[   0    0    0    1]
[   0    0 -q^2   -q]
[   0    0    0    0], 0: [1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1]}
this is the character, by coefficients of t^r
The following representation is the coefficient of t^1
This is a U-rep of dimension 4. E0,F0,K0,E1,F1,K1 act by 
E0:
[        0         0         0         0]
[        0         0         0         0]
[        q         1         0         0]
[q^4 - q^2        -q         0         0]
F0:
[            0             0         1/q^3         1/q^4]
[            0             0 (q^2 - 1)/q^2      (-1)/q^3]
[            0             0             0             0]
[            0             0           

In [12]:
monoidal_jantzen_filtration([3,5,1])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{1: [               0                0]
[               1                0]
[(-q^3)/(q^2 + 1)                0]
[               0                1]
[(-q^2)/(q^2 + 1)                0]
[               0              1/q]
[               0         -q^2 - 1]
[               0                0], 0: [  1   0   0   0   0   0]
[  0   1   0   0   0   0]
[  0   0   1   0   0   0]
[  0   0   0   1   0   0]
[  0   0 1/q   0   0   0]
[  0   0   0 1/q   0   0]
[  0   0   0   0   1   0]
[  0   0   0   0   0   1], -1: [1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1]}
this is the character, by coefficients of t^r
The following representation is the coefficient of t^1
This is a U-rep of dimension 2. E0,F0,K0,E1,F1,K1 act by 
E0:
[            0             0]
[q^5/(q^2 + 1)             0]


In [13]:
monoidal_jantzen_filtration([5,1,3])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{1: [               0                0]
[               1                0]
[             1/q                0]
[               0                1]
[        -q^2 - 1                0]
[               0 (-q^3)/(q^2 + 1)]
[               0 (-q^2)/(q^2 + 1)]
[               0                0], 0: [  1   0   0   0   0   0]
[  0   1   0   0   0   0]
[  0 1/q   0   0   0   0]
[  0   0   1   0   0   0]
[  0   0   0   1   0   0]
[  0   0   0   0   1   0]
[  0   0   0   0 1/q   0]
[  0   0   0   0   0   1], -1: [1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1]}
this is the character, by coefficients of t^r
The following representation is the coefficient of t^1
This is a U-rep of dimension 2. E0,F0,K0,E1,F1,K1 act by 
E0:
[          0           0]
[(q^2 + 1)/q           0]
F0:
[   

In [14]:
monoidal_jantzen_filtration([1,3,5])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{0: [    1     0     0     0]
[    0     1     0     0]
[    0   1/q     0     0]
[    0     0     1     0]
[    0 1/q^2     0     0]
[    0     0   1/q     0]
[    0     0 1/q^2     0]
[    0     0     0     1], -1: [1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1]}
this is the character, by coefficients of t^r
The following representation is the coefficient of t^0
This is a U-rep of dimension 4. E0,F0,K0,E1,F1,K1 act by 
E0:
[                  0                   0                   0                   0]
[                q^4                   0                   0                   0]
[                  0             q^3 + q                   0                   0]
[                  0                   0 (q^4 + q^2 + 1)/q^2                   0]
F0:
[                  0

In [15]:
monoidal_jantzen_filtration([3,3,1,1])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{4: [  0]
[  0]
[  0]
[  1]
[  0]
[ -q]
[  0]
[  0]
[  0]
[  0]
[ -q]
[  0]
[q^2]
[  0]
[  0]
[  0], 3: [ 0  0  0  0]
[ 0  0  0  0]
[ 1  0  0  0]
[ 0  1  0  0]
[-q  0  0  0]
[ 0 -q  0  0]
[ 0  0  0  0]
[ 0  0  0  0]
[ 0  0  0  0]
[ 0  0  0  0]
[ 0  0  1  0]
[ 0  0  0  1]
[ 0  0 -q  0]
[ 0  0  0 -q]
[ 0  0  0  0]
[ 0  0  0  0], 1: [       0        0        0        0        0        0        0]
[       1        0        0        0        0        0        0]
[       0        1        0        0        0        0        0]
[       0        0        1        0        0        0        0]
[-q^2 - 1       -q        0        0        0        0        0]
[       0        0       -q        0        0        0        0]
[       0        0        0        1        0        0        0]
[       0        0        0        0        1        0        0]
[       q        0       

In [16]:
monoidal_jantzen_filtration([3,1,3,1])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{3: [], 2: [  0]
[  0]
[  0]
[  0]
[  0]
[  1]
[ -q]
[  0]
[  0]
[ -q]
[q^2]
[  0]
[  0]
[  0]
[  0]
[  0], 1: [   0    0    0    0    0    0    0]
[   1    0    0    0    0    0    0]
[  -q    0    0    0    0    0    0]
[   0    0    0    0    0    0    0]
[   0    1    0    0    0    0    0]
[   0    0    1    0    0    0    0]
[   0    0    0    1    0    0    0]
[   0    0    0    0    1    0    0]
[   0   -q    0    0    0    0    0]
[   0    0    0    0    0    1    0]
[   0    0 -q^2   -q    0   -q    0]
[   0    0    0    0   -q    0    0]
[   0    0    0    0    0    0    0]
[   0    0    0    0    0    0    1]
[   0    0    0    0    0    0   -q]
[   0    0    0    0    0    0    0], 0: [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]


In [17]:
monoidal_jantzen_filtration([3,1,1,3])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{2: [  0   0   0]
[  0   0   0]
[  0   0   0]
[  0   0   0]
[  1   0   0]
[  0   1   0]
[  0 1/q   0]
[  0   0   1]
[ -q   0   0]
[  0  -q   0]
[  0  -1   0]
[  0   0  -q]
[  0   0   0]
[  0   0   0]
[  0   0   0]
[  0   0   0], 0: [  1   0   0   0   0   0   0   0   0   0   0   0   0]
[  0   1   0   0   0   0   0   0   0   0   0   0   0]
[  0 1/q   0   0   0   0   0   0   0   0   0   0   0]
[  0   0   1   0   0   0   0   0   0   0   0   0   0]
[  0   0   0   1   0   0   0   0   0   0   0   0   0]
[  0   0   0   0   1   0   0   0   0   0   0   0   0]
[  0   0   0   0   0   1   0   0   0   0   0   0   0]
[  0   0   0   0   0   0   1   0   0   0   0   0   0]
[  0   0   0   0   0   0   0   1   0   0   0   0   0]
[  0   0   0   0   0   0   0   0   1   0   0   0   0]
[  0   0   0   0   1  -q   0   0 1/q   0   0   0   0]
[  0   0   0   0   0   0   0   0   0   1   0   0   

In [18]:
monoidal_jantzen_filtration([1,3,3,1])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{2: [  0   0   0]
[  1   0   0]
[ -q   0   0]
[  0   0   0]
[  0   0   0]
[  0   1   0]
[  0  -q   0]
[  0   0   0]
[  0   0   0]
[  0 1/q   0]
[  0  -1   0]
[  0   0   0]
[  0   0   0]
[  0   0   1]
[  0   0  -q]
[  0   0   0], 0: [  1   0   0   0   0   0   0   0   0   0   0   0   0]
[  0   1   0   0   0   0   0   0   0   0   0   0   0]
[  0   0   1   0   0   0   0   0   0   0   0   0   0]
[  0   0   0   1   0   0   0   0   0   0   0   0   0]
[  0   0   0   0   1   0   0   0   0   0   0   0   0]
[  0   0   0   0   0   1   0   0   0   0   0   0   0]
[  0   0   0   0   0   0   1   0   0   0   0   0   0]
[  0   0   0   0   0   0   0   1   0   0   0   0   0]
[  0   0   0   0 1/q   0   0   0   0   0   0   0   0]
[  0   0   0   0   0   0   0   0   1   0   0   0   0]
[  0   0   0   0   0   1 1/q   0  -q   0   0   0   0]
[  0   0   0   0   0   0   0 1/q   0   0   0   0   

In [19]:
monoidal_jantzen_filtration([1,3,1,3])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{1: [], 0: [    1     0     0     0     0     0     0     0     0]
[    0     1     0     0     0     0     0     0     0]
[    0   1/q     0     0     0     0     0     0     0]
[    0     0     1     0     0     0     0     0     0]
[    0     0     0     1     0     0     0     0     0]
[    0     0     0     0     1     0     0     0     0]
[    0     0     0     0   1/q     0     0     0     0]
[    0     0     0     0     0     1     0     0     0]
[    0     0     0   1/q     0     0     0     0     0]
[    0     0     0     0   1/q     0     0     0     0]
[    0     0     0     0 1/q^2     0     0     0     0]
[    0     0     0     0     0   1/q     0     0     0]
[    0     0     0     0     0     0     1     0     0]
[    0     0     0     0     0     0     0     1     0]
[    0     0     0     0     0     0     0   1/q     0]
[    0     0     0     0  

In [20]:
monoidal_jantzen_filtration([1,1,3,3])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{0: [            1             0             0             0             0             0             0             0             0]
[            0             1             0             0             0             0             0             0             0]
[            0             0             1             0             0             0             0             0             0]
[            0             0             0             1             0             0             0             0             0]
[            0             0           1/q             0             0             0             0             0             0]
[            0             0             0           1/q             0             0             0             0             0]
[            0             0             0             0             1             0             0       

In [21]:
monoidal_jantzen_filtration([1,3,5,7])

computed rep
computed R matrices
computed descending decomposition
computed ascending decomposition
this is the filtration
{0: [    1     0     0     0     0]
[    0     1     0     0     0]
[    0   1/q     0     0     0]
[    0     0     1     0     0]
[    0 1/q^2     0     0     0]
[    0     0   1/q     0     0]
[    0     0 1/q^2     0     0]
[    0     0     0     1     0]
[    0 1/q^3     0     0     0]
[    0     0 1/q^2     0     0]
[    0     0 1/q^3     0     0]
[    0     0     0   1/q     0]
[    0     0 1/q^4     0     0]
[    0     0     0 1/q^2     0]
[    0     0     0 1/q^3     0]
[    0     0     0     0     1], -1: [       1        0        0        0        0        0        0        0        0        0        0        0        0        0        0]
[       0        1        0        0        0        0        0        0        0        0        0        0        0        0        0]
[       0        0        1        0        0        0        0        0        0 